# GreenRoot Nursery Co. — Reporting Dashboard (Colab version)

This is the Colab-adapted version of `01_reporting_dashboard.py`.
The only real difference from the local version: instead of relative paths
like `../data`, we upload the project zip and point `DATA`/`CHARTS` at
folders inside Colab's `/content` workspace.

### Step 1 — Upload the project zip
Run this cell, then choose `plant-nursery-analytics.zip` from your computer when the upload button appears.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select plant-nursery-analytics.zip in the dialog

### Step 2 — Unzip it into Colab's workspace

In [ ]:
import zipfile, os

with zipfile.ZipFile("plant-nursery-analytics.zip", "r") as z:
    z.extractall(".")

# confirm it landed where we expect
os.listdir("plant-nursery-analytics")

### Step 3 — Imports and path setup

This is the same block as the local script, EXCEPT the paths are now
absolute (pointing into the unzipped folder) instead of relative (`../data`).
Relative paths depend on which folder the script is *running from* — Colab
notebooks don't sit inside a `notebooks/` folder the way the local script
did, so `../data` would point to the wrong place. Absolute paths sidestep
that problem entirely.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

plt.style.use("seaborn-v0_8-whitegrid")

DATA = "plant-nursery-analytics/data"
CHARTS = "plant-nursery-analytics/charts"
os.makedirs(CHARTS, exist_ok=True)  # savefig won't create missing folders on its own

orders = pd.read_csv(f"{DATA}/orders.csv", parse_dates=["date"])
inventory = pd.read_csv(f"{DATA}/inventory.csv")
skus = pd.read_csv(f"{DATA}/skus.csv")

orders["revenue"] = orders["quantity"] * orders["unit_price"]
orders.head()

### KPI 1 — Weekly revenue & order volume trend

In [ ]:
weekly = orders.set_index("date").resample("W").agg(
    revenue=("revenue", "sum"), orders=("order_id", "count")
).reset_index()

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(weekly["date"], weekly["revenue"], color="#2e7d32", linewidth=2)
ax1.set_ylabel("Weekly Revenue ($)", color="#2e7d32")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))
ax2 = ax1.twinx()
ax2.plot(weekly["date"], weekly["orders"], color="#8d6e63", linewidth=1.3, alpha=0.7)
ax2.set_ylabel("Weekly Orders", color="#8d6e63")
ax1.set_title("Weekly Revenue & Order Volume — Spring/Fall Planting Season Peaks")
fig.tight_layout()
fig.savefig(f"{CHARTS}/01_weekly_revenue_trend.png", dpi=140)
plt.show()  # In Colab, plt.show() displays the chart inline — savefig alone won't

### KPI 2 — On-time delivery rate by category

In [ ]:
otd = orders.groupby("category")["on_time"].mean().sort_values() * 100
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(otd.index, otd.values, color="#558b2f")
ax.axvline(95, color="#c62828", linestyle="--", linewidth=1, label="95% SLA target")
ax.set_xlabel("On-Time Delivery Rate (%)")
ax.set_title("On-Time Delivery Rate by Category")
ax.set_xlim(75, 100)
ax.legend()
for b, v in zip(bars, otd.values):
    ax.text(v + 0.2, b.get_y() + b.get_height()/2, f"{v:.1f}%", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(f"{CHARTS}/02_otd_by_category.png", dpi=140)
plt.show()

### KPI 3 — Inventory health: on-hand stock vs. reorder point

In [ ]:
inv = inventory.copy()
inv["status"] = inv.apply(
    lambda r: "Below Reorder Point" if r.on_hand_units < r.reorder_point else "Healthy", axis=1
)
inv_sorted = inv.sort_values("on_hand_units")
colors = inv_sorted["status"].map({"Below Reorder Point": "#c62828", "Healthy": "#2e7d32"})

fig, ax = plt.subplots(figsize=(11, 10))
ax.barh(inv_sorted["product_name"], inv_sorted["on_hand_units"], color=colors, label="_nolegend_")
ax.scatter(inv_sorted["reorder_point"], inv_sorted["product_name"], color="black", marker="|", s=100, label="Reorder point")
ax.set_xlabel("Units")
ax.set_title("Inventory On-Hand vs. Reorder Point (red = needs replenishment)")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(f"{CHARTS}/03_inventory_health.png", dpi=140)
plt.show()

### KPI 4 — Revenue mix by category

In [ ]:
cat_rev = orders.groupby("category")["revenue"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
ax.pie(cat_rev.values, labels=cat_rev.index, autopct="%1.0f%%", startangle=90,
       colors=plt.cm.Greens_r(range(60, 220, 25)))
ax.set_title("Revenue Mix by Category (18-mo)")
fig.tight_layout()
fig.savefig(f"{CHARTS}/04_revenue_mix.png", dpi=140)
plt.show()

### Summary stats

In [ ]:
total_revenue = orders["revenue"].sum()
otd_overall = orders["on_time"].mean() * 100
below_reorder = inv["status"].eq("Below Reorder Point").sum()
avg_review = orders["review_score"].mean()

print("=== OPS HEALTH SUMMARY ===")
print(f"Total revenue (18mo): ${total_revenue:,.0f}")
print(f"Overall on-time delivery rate: {otd_overall:.1f}%")
print(f"SKUs below reorder point: {below_reorder} / {len(inv)}")
print(f"Average review score: {avg_review:.2f} / 5")

### Step 4 (optional) — download the generated charts back to your computer

In [ ]:
import shutil
from google.colab import files as colab_files

shutil.make_archive("charts_output", "zip", CHARTS)
colab_files.download("charts_output.zip")